## Scenario 1: A single data scientist participating in an ML competition

MLflow setup:
* Tracking server: no
* Backend store: local filesystem
* Artifacts store: local filesystem

The experiments can be explored locally by launching the MLflow UI.

In [1]:
import mlflow

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'sqlite:////workspaces/MLOPs/02-mlflow/mlflow.db'


In [6]:
mlflow.search_experiments()

[<Experiment: artifact_location='/workspaces/MLOPs/02-mlflow/mlruns/2', creation_time=1789216623767, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1789216623767, lifecycle_stage='active', name='test', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/MLOPs/02-mlflow/mlruns/1', creation_time=1789209710348, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789209710348, lifecycle_stage='active', name='iris-classification', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/MLOPs/02-mlflow/mlruns/0', creation_time=1789209710342, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1789209710342, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Creating an experiment and logging a new run

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/09/13 11:32:31 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/09/13 11:32:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


default artifacts URI: '/workspaces/MLOPs/02-mlflow/mlruns/3/a9632baefe2b4eae8a79f662c966ef34/artifacts'


In [8]:
mlflow.search_experiments()

[<Experiment: artifact_location='/workspaces/MLOPs/02-mlflow/mlruns/3', creation_time=1789299151632, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1789299151632, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/MLOPs/02-mlflow/mlruns/2', creation_time=1789216623767, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1789216623767, lifecycle_stage='active', name='test', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/MLOPs/02-mlflow/mlruns/1', creation_time=1789209710348, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789209710348, lifecycle_stage='active', name='iris-classification', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/MLOPs/02-mlflow/mlruns/0', creation_time=1789209710342, effective_trace_archi

### Interacting with the model registry

In [9]:
from mlflow.tracking import MlflowClient



client = MlflowClient()

In [10]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException:
    print("It's not possible to access the model registry :(")